# Real-time order tracking project

Three waves of order events arrive as files. After each wave the tracking stream reads only
the new files and updates three stores:

* Iceberg `ice.tracking.order_events`: append-only event log
* Delta `order_status`: current status per order (MERGE, newest event wins)
* Postgres `order_tracking`: serving table for an app

Wave 3 has a duplicate and a late event on purpose. The SLA report checks that all three
stores agree.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
steps = ["orders-wave1", "orders-stream1", "orders-wave2", "orders-stream2",
         "orders-wave3", "orders-stream3", "orders-report"]
summary = lab.run_many(steps, run_id=RID, stop_on_failure=True)
summary

In [ ]:
lab.pg("SELECT status, count(*) FROM order_tracking GROUP BY 1 ORDER BY 2 DESC")

In [ ]:
lab.sql("SELECT status, count(*) n FROM default.order_status GROUP BY 1 ORDER BY 2 DESC", lake=True)

In [ ]:
lab.sql("SELECT event_type, count(*) n FROM ice.tracking.order_events GROUP BY 1 ORDER BY 2 DESC", lake=True)

One snapshot per micro-batch that added data, so you can see each wave land:

In [ ]:
lab.sql("""SELECT committed_at, operation, summary['added-records'] AS added
           FROM ice.tracking.order_events.snapshots ORDER BY committed_at""", lake=True)

Follow one order through its events:

In [ ]:
oid = lab.pg("SELECT order_id FROM order_tracking ORDER BY events_seen DESC LIMIT 1")["order_id"][0]
lab.sql(f"SELECT event_time, event_type, city, carrier FROM ice.tracking.order_events WHERE order_id = '{oid}' ORDER BY event_time", lake=True)